# E8–E9: meanings as interdependent relations
CPU. Hypothesis: interpret observations through a jointly inferred relational system; later evidence changes inferred meanings of all earlier relations. Inspired by Bhartṛhari's Vākyapadīya 1.84–85 ([1.84](https://www.wisdomlib.org/hinduism/book/vakyapadiya-of-bhartrihari/d/doc1216529.html), [1.85](https://www.wisdomlib.org/hinduism/book/vakyapadiya-of-bhartrihari/d/doc1216530.html)). Not a claim about an ancient learning algorithm.

E8 supplies two possible algebras, Z4 and the Klein four-group, but not the meaning of each relation symbol or the latent state of any object. Fit both jointly from evidence and allow a no-structure hypothesis. Prediction: the mixture should nearly match the correct fixed algebra and the null should prevent inventing relationships in random worlds. This is standard Bayesian model averaging, not a novel recursive learning algorithm. Incremental posterior updates are equivalent to exact full-history inference.

Related controls for a future real-world test: [higher-order logic learning](https://ijcai.org/proceedings/2016/204), [OPINE-World](https://arxiv.org/abs/2607.01531), [WARC](https://arxiv.org/abs/2611.12324). No GPU or SOTA claim.

In [ ]:
import os
os.environ['OPENBLAS_NUM_THREADS']='1'  # before NumPy import in fresh kernel
import numpy as np
from itertools import product, permutations
from time import perf_counter
N,K=5,4
PAIRS=np.array([(i,j) for i in range(N) for j in range(N) if i!=j]); Q=len(PAIRS)
Z=np.array([(0,*x) for x in product(range(K),repeat=N-1)])
MAP=np.array(list(permutations(range(K))))
H=[]; TABLE=[]
for kind in ('cyclic','xor'):
    a,b=Z[:,PAIRS[:,0]],Z[:,PAIRS[:,1]]
    d=(b-a)%K if kind=='cyclic' else a^b
    H.append(MAP[:,d].reshape(-1,Q))
    x,y=np.indices((K,K)); op=(x+y)%K if kind=='cyclic' else x^y
    TABLE.append(np.array([m[op[np.argsort(m)[:,None],np.argsort(m)[None,:]]] for m in MAP]))
H=np.array(H,dtype=np.uint8); TABLE=np.array(TABLE)
ONE=np.eye(K,dtype=np.float32)[H].reshape(2,H.shape[1],-1)
EPS=.05
assert H.shape==(2,6144,20)

def trial(rng,kind):
    truth=H[kind,rng.integers(H.shape[1])].copy() if kind<2 else rng.integers(K,size=Q)
    order=rng.permutation(Q); w=np.ones(H.shape[:2]); rows=[]
    for t,j in enumerate(order[:16],1):
        y=truth[j] if rng.random()>=EPS else (truth[j]+rng.integers(1,K))%K
        w*=np.where(H[:,:,j]==y,1-EPS,EPS/(K-1))
        if t not in (4,8,12,16):continue
        mass=w.mean(1); posterior=w/w.sum(1,keepdims=True)
        pred=np.stack([posterior[g]@ONE[g] for g in range(2)]).reshape(2,Q,K)
        uniform=np.full((Q,K),1/K)
        mix=np.einsum('g,gqk->qk',mass/mass.sum(),pred)
        allmass=np.r_[mass,(1/K)**t]
        robust=(mass[0]*pred[0]+mass[1]*pred[1]+allmass[2]*uniform)/allmass.sum()
        unseen=order[t:]; target=np.eye(K)[truth[unseen]]
        rows.append([np.mean(np.sum((p[unseen]-target)**2,axis=1)) for p in (uniform,*pred,mix,robust)])
    return rows

Score multiclass Brier error on unobserved ordered pairs; lower is better. Independent seeds, 150 worlds per family/seed; all arms see identical data. The fixed-correct algebra is a strong oracle-family comparator. Predictions after 4/8/12/16 observations. The null has no relational structure. The generator shares the candidate families by construction.

In [ ]:
start=perf_counter()
for seed in (42,173):
    rng=np.random.default_rng(seed)
    for kind in range(3):
        v=np.array([trial(rng,kind) for _ in range(150)])
        print(seed,('cyclic','xor','random')[kind],np.round(v.mean(0),4))
        d=v[:,2,3]-v[:,2,4];boot=rng.choice(d,(3000,len(d)),replace=True).mean(1)
        print('robust improvement vs mix @12',round(d.mean(),4),np.round(np.quantile(boot,[.025,.975]),4))
print('seconds',perf_counter()-start)

Standalone CPU run, 900 worlds, 7.79 s. Mean across both seeds at 12 observations:

| World | Uniform | Correct fixed algebra | Two-algebra mixture | Mixture + null |
|---|---:|---:|---:|---:|
| Cyclic | .7500 | .02235 | .0407 | .0513 |
| XOR | .7500 | .0252 | .0360 | .04105 |
| Random | .7500 | N/A | 1.13325 | .79635 |

Null improves random performance relative to forced structure; it does not match the uniform oracle and costs some structured performance. Correct fixed algebra wins on structured tasks. No novel recursive mechanism established. No held-out domain transfer: both algebras are supplied, and the random world is modeled by the null.

## E9 — Does uncertainty about meaning guide useful acquisition?
All arms use exactly the E8 robust posterior. Reserve six test pairs; ask up to twelve questions from the other fourteen. Compare random, outcome entropy, mutual information with the inferred composition table (including the null), and a hybrid. Hypothesis before testing: semantic information gain might clarify multiple relations at once and therefore reduce held-out prediction error faster. It may instead clarify irrelevant abstractions.

In [ ]:
tables,idx=np.unique(TABLE.reshape(-1,K*K),axis=0,return_inverse=True)
G=np.repeat(idx,len(Z)); nh=H.shape[1]*2; F=ONE.reshape(nh,-1); HH=H.reshape(nh,Q)
S=np.eye(len(tables),dtype=np.float32)[G].T.copy()
def ent(p):return -np.sum(p*np.log(np.maximum(p,1e-30)),axis=-1)
def episode(seed,kind,policy):
    rng=np.random.default_rng(seed)
    truth=H[kind,rng.integers(H.shape[1])].copy() if kind<2 else rng.integers(K,size=Q)
    order=rng.permutation(Q); test=order[:6]; pool=list(order[6:])
    observed=(truth+(rng.random(Q)<EPS)*rng.integers(1,K,Q))%K
    w=np.full(nh,1/(3*H.shape[1])); null=1/3; out=[]
    for t in range(1,13):
        norm=w.sum()+null; wn=w/norm; nn=null/norm
        p=(wn@F).reshape(Q,K)+nn/K; noisy=EPS/(K-1)+(1-EPS-EPS/(K-1))*p
        if policy==0:j=pool[0]
        else:
            uncertainty=ent(noisy)
            if policy==1: score=uncertainty
            else:
                sem=((S*wn)@F).reshape(len(tables),Q,K); mass=sem.sum(-1,keepdims=True)
                cond=EPS/(K-1)*mass+(1-EPS-EPS/(K-1))*sem
                mi=uncertainty-(mass[...,0]*ent(cond/np.maximum(mass,1e-30))).sum(0)-nn*np.log(K)
                score=mi if policy==2 else mi+.25*uncertainty
            j=max(pool,key=lambda x:score[x])
        pool.remove(j);w*=np.where(HH[:,j]==observed[j],1-EPS,EPS/(K-1));null/=K
        if t in (4,8,12):
            p=((w@F).reshape(Q,K)+null/K)/(w.sum()+null)
            out.append(np.mean(np.sum((p[test]-np.eye(K)[truth[test]])**2,axis=1)))
    return out

In [ ]:
start=perf_counter()
for kind in range(3):
    for seed in (1000,2000):
        v=np.array([[episode(seed+i,kind,p) for p in range(4)] for i in range(40)])
        print(kind,seed,np.round(v.mean(0),4))
        d=v[:,1,1]-v[:,2,1];boot=np.random.default_rng(41).choice(d,(3000,len(d)),replace=True).mean(1)
        print('semantic vs uncertainty at 8',round(d.mean(),4),np.round(np.quantile(boot,[.025,.975]),4))
print('seconds',perf_counter()-start)

Standalone CPU run: 240 worlds, four policies per world, 27.41 s. Means across seeds at 8 observations:

| World | Random | Outcome entropy | Semantic MI | Hybrid |
|---|---:|---:|---:|---:|
| Cyclic | .21685 | .1476 | .37725 | .2611 |
| XOR | .1650 | .11315 | .35485 | .19425 |
| Random | .8329 | .88855 | .78955 | .8268 |

All four structured seed comparisons favor outcome entropy, with paired bootstrap intervals excluding zero. Semantic querying wins on random worlds relative to uncertainty sampling but is far worse in the structured worlds where we expected benefits. No wall-clock matching (semantic querying costs more). Reject semantic MI alone as an acquisition policy. No claim of novelty or GPU-worthy result.

Next bet: learn the hypothesis language itself from diverse observations, not just choose between supplied algebras. Require independent task generators, non-group and changing semantics, held-out compositional tasks, matched budgets, and a strong relational induction baseline. On actual WARC recordings compare with full test-time training and simple retrieval; do not equate replay scores with official MechEval. GPU gate is a reproducible real-data improvement or a strong public benchmark result, not another engineered toy success.